<a href="https://colab.research.google.com/github/sairiamu/ML-DL-collection/blob/main/BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install transformers torch

In [3]:
from transformers import pipeline

# Initialize the pipeline for fill-mask using a BERT model
bert = pipeline("fill-mask", model="bert-base-uncased")

text = "The patient has a [MASK] and needs an appointment."

results = bert(text, top_k=5)

for result in results:
    print(
        result["token_str"],
        round(result["score"], 4)
    )

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

headache 0.235
problem 0.0951
fever 0.0522
concussion 0.0451
cold 0.0449


In [4]:

examples = [
    "The patient needs a [MASK] appointment.",
    "Please record the patient's [MASK].",
    "The clinic has scheduled a [MASK] visit.",
    "The patient reported a [MASK] yesterday."
]

for sentence in examples:
    print("\nInput:", sentence)

    for item in bert(sentence, top_k=3):
        print("  Suggestion:", item["token_str"],
              "| Score:", round(item["score"], 3))



Input: The patient needs a [MASK] appointment.
  Suggestion: medical | Score: 0.196
  Suggestion: dentist | Score: 0.098
  Suggestion: dental | Score: 0.083

Input: Please record the patient's [MASK].
  Suggestion: name | Score: 0.058
  Suggestion: status | Score: 0.058
  Suggestion: condition | Score: 0.053

Input: The clinic has scheduled a [MASK] visit.
  Suggestion: second | Score: 0.142
  Suggestion: special | Score: 0.032
  Suggestion: surprise | Score: 0.032

Input: The patient reported a [MASK] yesterday.
  Suggestion: concussion | Score: 0.284
  Suggestion: stroke | Score: 0.181
  Suggestion: fever | Score: 0.093


In [5]:

def clinicbert_suggest(text, top_k=5):
    if not isinstance(text, str) or not text.strip():
        return []

    # BERT expects one mask token for this prediction
    if text.count("[MASK]") != 1:
        raise ValueError(
            "Please include exactly one [MASK] token."
        )

    predictions = bert(
        text,
        top_k=max(1, min(top_k, 10))
    )

    return [
        {
            "suggestion": item["token_str"].strip(),
            "score": round(float(item["score"]), 4)
        }
        for item in predictions
    ]


# Test the function
clinicbert_suggest(
    "The patient needs a [MASK] appointment."
)


[{'suggestion': 'medical', 'score': 0.1955},
 {'suggestion': 'dentist', 'score': 0.0981},
 {'suggestion': 'dental', 'score': 0.0831},
 {'suggestion': 'hospital', 'score': 0.0445},
 {'suggestion': 'new', 'score': 0.0406}]